# Task Notebook

## Goal
Run a reproducible task estimate-vs-actual analysis and emit a JSON artifact.

## Done Looks Like
- Inputs are explicit and validated against schema v1.
- Analysis computes aggregate metrics and largest positive delta details.
- Output artifact is written to `output/artifacts/summary.json`.


## 1) Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

# Make notebook import robust whether opened from repo root or notebook directory.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "task_analysis").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate repository root containing task_analysis package.")

from task_analysis.core import (
    ValidationError,
    analyze_task_items,
    build_takeaways,
    parse_task_items,
    save_summary,
)


## 2) Inputs

Input schema v1:
- `name`: non-empty string
- `estimate_hours`: number or numeric string (when coercion is enabled), must be `>= 0`
- `actual_hours`: number or numeric string (when coercion is enabled), must be `>= 0`


In [ ]:
# Edit this list for your task.
raw_items = [
    {"name": "Plan", "estimate_hours": 1.5, "actual_hours": 2.0},
    {"name": "Implement", "estimate_hours": 3.0, "actual_hours": 2.5},
    {"name": "Validate", "estimate_hours": 1.0, "actual_hours": 1.2},
]

coerce_numeric = True

try:
    items = parse_task_items(raw_items, coerce_numeric=coerce_numeric)
except ValidationError as exc:
    raise ValueError(f"Input validation failed: {exc}") from exc

len(items)


## 3) Analysis


In [ ]:
summary = analyze_task_items(items)
summary


## 4) Concise Takeaways + Artifact


In [ ]:
takeaways = build_takeaways(summary)

for line in takeaways:
    print(f"- {line}")

artifact_path = Path("output/artifacts/summary.json")
save_summary(summary, artifact_path)
print(f"Saved summary artifact to {artifact_path}")
